# Behavioural Analysis and User Segmentation

Analysing rider and driver behaviour patterns and creating interpretable segments.

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

from roadies.ingestion.loaders import load_csv
from roadies.features.demand_supply import engineer_demand_supply_features
from roadies.features.surge import engineer_surge_features
from roadies.features.acceptance import engineer_acceptance_features
from roadies.features.cancellation import engineer_cancellation_features
from roadies.features.experience import engineer_experience_features
from roadies.features.demand_period import classify_high_demand
from roadies.analysis.behavioral import (
    analyze_rider_behaviour,
    segment_riders,
    summarize_rider_segments,
    analyze_driver_behaviour,
    segment_drivers,
    summarize_driver_segments,
    compare_behaviour_by_demand,
    analyze_repeated_behaviour,
)

In [ ]:
# Load and engineer features
df = load_csv(Path('data/raw/rides.csv'))
df, _ = engineer_demand_supply_features(df)
df, _ = engineer_surge_features(df)
df, _ = engineer_acceptance_features(df)
df, _ = engineer_cancellation_features(df)
df, _ = engineer_experience_features(df)
df, _ = classify_high_demand(df)
print(f'Dataset: {len(df)} rows')

In [ ]:
# Rider segments
rider_summaries = summarize_rider_segments(df)
print('Rider segments:')
for s in rider_summaries:
    print(f'  {s.segment_name}: {s.user_count} riders, cancel={s.cancellation_rate:.1%}')

In [ ]:
# Driver segments
driver_summaries = summarize_driver_segments(df)
print('Driver segments:')
for s in driver_summaries:
    print(f'  {s.segment_name}: {s.user_count} drivers, acceptance={s.acceptance_rate:.1%}')

In [ ]:
# High-demand comparison
comparison = compare_behaviour_by_demand(df)
print('High-demand vs normal:')
for metric, change in comparison.get('change_pct', {}).items():
    print(f'  {metric}: {change:+.1%}')

In [ ]:
# Visualise rider segments
riders = segment_riders(df)
segment_cols = ['cancellation_sensitive', 'completion_oriented', 'high_wait_exposure', 'high_surge_exposure']
sizes = {col: riders[col].sum() for col in segment_cols if col in riders.columns}
px.bar(x=list(sizes.keys()), y=list(sizes.values()), title='Rider Segment Sizes')